# TDBM Extreme-Sample GIFs (AV2 + Waymo)

Visualize sample agents from each TDBM class as GIFs with the same speed / relative-speed / long-accel / lat-accel dashboard used for our main pipeline and the KDSC replication.

- Loads labels produced by `tdbm_replication.ipynb` (`artifacts/tdbm_replication/<dataset>/{dataset}_val_tdbm.parquet`).
- Uses the same custom scene renderer as the Waymo / KDSC GIF notebooks (only the center agent is red).
- TDBM classifies by `argmax` over per-row scores, so 'extremeness' is the **score margin** = `own_score - max(other_scores)`. Higher margin = more confidently classified in its class.

Two label modes via the `LABEL_COL` switch:

- `tdbm_raw_style_label`: 6-way (Aggressive / Reckless / Threatening / Careful / Cautious / Timid)
- `style_label_tdbm`:     binary (aggressive / normal)

Outputs go to `artifacts/tdbm_replication/<dataset>/figures/gifs/`.

In [1]:
from __future__ import annotations
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd

# -------- DATASET / LABEL SWITCH --------
DATASET    = 'av2'                  # 'av2' or 'waymo'
SPLIT      = 'val'                  # waymo uses 'val' too via the CATALOG default
LABEL_COL  = 'tdbm_raw_style_label' # or 'style_label_tdbm' for binary
PER_CLUSTER = 5
RANDOM_STATE = 42
# ----------------------------------------

_CWD = Path.cwd()
for cand in (_CWD, _CWD.parent, _CWD.parent.parent):
    if (cand / 'src').is_dir():
        REPO_ROOT = cand
        break
else:
    REPO_ROOT = _CWD
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from tailrisk_mp.runtime import ensure_numpy_pickle_compat
ensure_numpy_pickle_compat()

TDBM_DIR = REPO_ROOT / 'artifacts' / 'tdbm_replication' / DATASET
GIF_DIR  = TDBM_DIR / 'figures' / 'gifs'
GIF_DIR.mkdir(parents=True, exist_ok=True)

VAL_PARQUET = TDBM_DIR / f'{DATASET}_val_tdbm.parquet'
if not VAL_PARQUET.exists():
    raise FileNotFoundError(f'Missing {VAL_PARQUET} - run tdbm_replication.ipynb (DATASET={DATASET!r}) first.')

val = pd.read_parquet(VAL_PARQUET)
SCORE_COLS = [f'tdbm_score_{k}' for k in range(6)]
missing = [c for c in SCORE_COLS + [LABEL_COL, 'tdbm_raw_style_id'] if c not in val.columns]
if missing:
    raise KeyError(f'TDBM parquet missing columns: {missing}')

print('dataset      :', DATASET)
print('val rows     :', len(val))
print('label column :', LABEL_COL)
print('class sizes  :'); print(val[LABEL_COL].value_counts())

dataset      : av2
val rows     : 21809
label column : tdbm_raw_style_label
class sizes  :
tdbm_raw_style_label
Careful       21687
Timid           119
Aggressive        3
Name: count, dtype: int64


## 1. Score margin ('extremeness') per row

For TDBM the natural confidence signal is the score gap between the predicted class and the runner-up: `margin = score[predicted] - max(score[other])`. Larger margin = more decisively classified.

In [2]:
scores = val[SCORE_COLS].to_numpy(dtype=np.float32)        # (N, 6)
raw_id = val['tdbm_raw_style_id'].to_numpy(dtype=np.int64) # (N,)

own = scores[np.arange(len(scores)), raw_id]
masked = scores.copy()
masked[np.arange(len(scores)), raw_id] = -np.inf
runner_up = masked.max(axis=1)

val = val.copy()
val['tdbm_margin'] = own - runner_up
val[[LABEL_COL, 'tdbm_margin']].groupby(LABEL_COL).describe().round(2)

tdbm_margin                                        \
                           count    mean     std   min     25%     50%   
tdbm_raw_style_label                                                     
Aggressive                   3.0   44.55   45.76  3.48   19.89   36.30   
Careful                  21687.0  330.37  288.14  0.17  122.80  249.53   
Timid                      119.0    9.69   10.18  0.17    2.66    6.01   

                                       
                         75%      max  
tdbm_raw_style_label                   
Aggressive             65.09    93.88  
Careful               451.97  2481.63  
Timid                  12.90    48.01

## 2. Pick samples per class

Random sampling per class (same default as our other GIF notebooks). Swap to `nlargest('tdbm_margin')` for the most confidently classified rows or `nsmallest('tdbm_margin')` for the boundary cases.

In [3]:
picks: dict[str, pd.DataFrame] = {}
for label in sorted(val[LABEL_COL].dropna().unique()):
    sub = val[val[LABEL_COL] == label].dropna(subset=['scenario_id', 'center_objects_id'])
    if len(sub) == 0:
        continue
    top = sub.sample(min(PER_CLUSTER, len(sub)), random_state=RANDOM_STATE)
    # top = sub.nlargest(PER_CLUSTER, 'tdbm_margin')
    picks[label] = top
    print(f'\n=== {label} ({len(top)} samples, of {len(sub)} total) ===')
    cols = ['scenario_id', 'center_objects_id', 'tdbm_margin',
            'tdbm_v_avg', 'tdbm_j_l', 'tdbm_v_nei', 'tdbm_s_front',
            'avg_speed', 'max_abs_accel', 'max_abs_jerk']
    cols = [c for c in cols if c in top.columns]
    print(top[cols].round(3).to_string(index=False))


=== Aggressive (3 samples, of 3 total) ===
                         scenario_id center_objects_id  tdbm_margin  tdbm_v_avg  tdbm_j_l  tdbm_v_nei  tdbm_s_front  avg_speed  max_abs_accel  max_abs_jerk
76bfcc82-0598-4e1d-a434-3a2b237813e6             10774    93.883003      10.185   251.589       1.180        81.983     10.310          3.129        45.697
8628c1e9-f333-4ff2-bf10-f43ad8a5e1a3             25759     3.481000       2.564     8.217       1.828         3.564      0.530         17.984       148.443
909572a1-9d3d-4917-9297-863506e0f833             95250    36.296001       2.665    56.917      -3.194         0.445      2.351          2.441        16.618

=== Careful (5 samples, of 21687 total) ===
                         scenario_id center_objects_id  tdbm_margin  tdbm_v_avg  tdbm_j_l  tdbm_v_nei  tdbm_s_front  avg_speed  max_abs_accel  max_abs_jerk
848f757b-59b7-467b-8c4a-e5c04d8d8e43             92990    36.948002      10.296     7.868     -71.057         3.443     10.615     

## 3. Renderer (custom: only center red)

Same dashboard renderer used by the Waymo and KDSC GIF notebooks. Works for both AV2 and Waymo because every non-center agent is colored by track-type only.

In [4]:
import io
import matplotlib.pyplot as plt
from tailrisk_mp.scenario_viz import (
    load_scenario_by_id, _normalize_track_id, _map_style, _track_color,
    _valid_segments, _scenario_center,
)

DT  = 0.1
G   = 9.81
KMH = 3.6

SPEED_SMOOTH_FRAMES = 9
ACCEL_SMOOTH_FRAMES = 1

V_MAX_KMH       = 140.0
SPEED_WARN_KMH  =  60.0
SPEED_LIMIT_KMH =  90.0

V_REL_MAX_KMH    = 50.0
REL_WARN_KMH     = 15.0
REL_LIMIT_KMH    = 30.0

NEIGHBOR_RADIUS_M    = 30.0
NEIGHBOR_LAT_M       =  4.0
NEIGHBOR_DIR_DEG     = 30.0

A_LONG_MAX_G    = 3.0
ACC_LONG_WARN_G = 0.5
ACC_LONG_LIMIT_G= 0.8

A_LAT_MAX_G     = 1.5
ACC_LAT_WARN_G  = 0.3
ACC_LAT_LIMIT_G = 0.6

GREEN, ORANGE, RED = '#2ca02c', '#ff7f0e', '#d62728'

def _draw_scene_frame(ax, scenario, frame, center_track_id, view_radius):
    ax.clear()
    center_track_norm = _normalize_track_id(center_track_id)

    for feature in scenario['map_features'].values():
        poly = np.asarray(feature.get('polyline', []))
        if poly.ndim != 2 or len(poly) < 2:
            continue
        style = _map_style(feature.get('type', 'UNKNOWN'))
        ax.plot(poly[:, 0], poly[:, 1], color=style['color'],
                linewidth=style['linewidth'], zorder=1)

    tracks = scenario['tracks']
    center_xy = _scenario_center(scenario)
    for tid, t in tracks.items():
        if _normalize_track_id(tid) != center_track_norm and str(tid) != str(center_track_id):
            continue
        pos = np.asarray(t['state']['position'])
        valid = np.asarray(t['state']['valid']).reshape(-1).astype(bool)
        if frame < len(valid) and valid[frame]:
            center_xy = pos[frame, :2]
        break

    for track_id, track in tracks.items():
        state = track['state']
        positions = np.asarray(state['position'])
        valid = np.asarray(state['valid']).reshape(-1).astype(bool)
        norm_id = _normalize_track_id(track_id)
        is_center = norm_id == center_track_norm or str(track_id) == str(center_track_id)

        if is_center:
            color = RED
            line_width = 3.0
        else:
            color = _track_color(track.get('type', 'UNKNOWN'), 'other')
            line_width = 1.4

        history = valid.copy(); history[frame + 1:] = False
        future  = valid.copy(); future[: frame + 1] = False

        for segment in _valid_segments(positions, history):
            ax.plot(segment[:, 0], segment[:, 1], color=color, linewidth=line_width, zorder=3)
        for segment in _valid_segments(positions, future):
            ax.plot(segment[:, 0], segment[:, 1],
                    color=color, linewidth=max(1.0, line_width - 0.6),
                    linestyle='--', alpha=0.4, zorder=2)
        if frame < len(valid) and valid[frame]:
            xy = positions[frame, :2]
            ax.scatter(xy[0], xy[1], color=color,
                       s=90 if is_center else 18,
                       zorder=5, edgecolors='black' if is_center else 'none', linewidths=0.8)

    ax.set_aspect('equal')
    ax.set_xlim(center_xy[0] - view_radius, center_xy[0] + view_radius)
    ax.set_ylim(center_xy[1] - view_radius, center_xy[1] + view_radius)
    ax.axis('off')

def _zone_color(value, warn, red):
    a = abs(float(value))
    if a >= red:  return RED
    if a >= warn: return ORANGE
    return GREEN

def _smooth(arr, window):
    a = np.asarray(arr, dtype=float)
    if window <= 1 or a.size == 0:
        return a
    if a.ndim == 1:
        out = np.full_like(a, np.nan)
        half = window // 2
        T = len(a)
        for i in range(T):
            lo, hi = max(0, i - half), min(T, i + half + 1)
            seg = a[lo:hi]
            seg = seg[~np.isnan(seg)]
            if seg.size:
                out[i] = seg.mean()
        return out
    return np.stack([_smooth(a[:, j], window) for j in range(a.shape[1])], axis=1)

def _per_track_kinematics(positions, valid):
    T = len(valid)
    speed = np.full(T, np.nan)
    velocity = np.full((T, 2), np.nan)
    for i in range(1, T):
        if valid[i] and valid[i-1]:
            v = (positions[i] - positions[i-1]) / DT
            speed[i] = float(np.linalg.norm(v))
            velocity[i] = v
    return speed, velocity

def _compute_2d_accel(velocity):
    T = len(velocity)
    accel = np.full((T, 2), np.nan)
    for i in range(1, T):
        if not (np.any(np.isnan(velocity[i])) or np.any(np.isnan(velocity[i-1]))):
            accel[i] = (velocity[i] - velocity[i-1]) / DT
    return accel

def _decompose_long_lat(accel_vec, velocity):
    T = len(accel_vec)
    a_long = np.full(T, np.nan)
    a_lat  = np.full(T, np.nan)
    for i in range(T):
        if np.any(np.isnan(accel_vec[i])) or np.any(np.isnan(velocity[i])):
            continue
        spd = float(np.linalg.norm(velocity[i]))
        if spd < 0.1:
            continue
        h = velocity[i] / spd
        h_perp = np.array([-h[1], h[0]])
        a_long[i] = float(np.dot(accel_vec[i], h))
        a_lat[i]  = float(np.dot(accel_vec[i], h_perp))
    return a_long, a_lat

def _compute_relative_speed(scenario, ego_track, ego_pos, ego_valid, ego_speed, ego_vel):
    cos_thresh = float(np.cos(np.deg2rad(NEIGHBOR_DIR_DEG)))
    others = []
    for tid, t in scenario['tracks'].items():
        if t is ego_track:
            continue
        pos = np.asarray(t['state']['position'])[:, :2]
        v   = np.asarray(t['state']['valid']).reshape(-1).astype(bool)
        if pos.shape[0] < 2:
            continue
        s, vel = _per_track_kinematics(pos, v)
        s   = _smooth(s,   SPEED_SMOOTH_FRAMES)
        vel = _smooth(vel, SPEED_SMOOTH_FRAMES)
        others.append((pos, v, s, vel))

    T = len(ego_valid)
    rel = np.full(T, np.nan)
    for f in range(T):
        if not ego_valid[f] or np.isnan(ego_speed[f]):
            continue
        if ego_speed[f] < 1e-3 or np.any(np.isnan(ego_vel[f])):
            heading = np.array([1.0, 0.0])
        else:
            heading = ego_vel[f] / max(np.linalg.norm(ego_vel[f]), 1e-6)
        normal = np.array([-heading[1], heading[0]])

        neighbour_speeds: list[float] = []
        for pos, v, s, vel in others:
            if f >= len(v) or not v[f] or np.isnan(s[f]) or s[f] < 1e-3:
                continue
            d = pos[f] - ego_pos[f]
            if np.linalg.norm(d) > NEIGHBOR_RADIUS_M:
                continue
            if abs(float(np.dot(d, normal))) > NEIGHBOR_LAT_M:
                continue
            n_norm = max(np.linalg.norm(vel[f]), 1e-6)
            if float(np.dot(vel[f] / n_norm, heading)) < cos_thresh:
                continue
            neighbour_speeds.append(float(s[f]))

        rel[f] = (ego_speed[f] - float(np.mean(neighbour_speeds))) if neighbour_speeds else 0.0
    return rel

def _draw_speedometer(ax, speed_ms):
    ax.clear()
    speed_kmh = float(np.nan_to_num(speed_ms, nan=0.0)) * KMH
    theta = np.linspace(np.pi, 0, 200)
    speeds_at_theta = V_MAX_KMH * (np.pi - theta) / np.pi
    for i in range(len(theta) - 1):
        c = _zone_color(speeds_at_theta[i], SPEED_WARN_KMH, SPEED_LIMIT_KMH)
        ax.plot([np.cos(theta[i]), np.cos(theta[i+1])],
                [np.sin(theta[i]), np.sin(theta[i+1])],
                color=c, linewidth=10, solid_capstyle='butt')
    s_c = max(0.0, min(speed_kmh, V_MAX_KMH))
    angle = np.pi - np.pi * (s_c / V_MAX_KMH)
    ax.plot([0, 0.85 * np.cos(angle)], [0, 0.85 * np.sin(angle)],
            color='black', linewidth=3)
    ax.scatter([0], [0], s=40, color='black', zorder=5)
    color = _zone_color(speed_kmh, SPEED_WARN_KMH, SPEED_LIMIT_KMH)
    label = '--' if np.isnan(speed_ms) else f'{speed_kmh:.0f}'
    ax.text(0, -0.35, f'speed\n{label} km/h', ha='center', va='center',
            fontsize=10, color=color, fontweight='bold')
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.6, 1.2)
    ax.set_aspect('equal'); ax.axis('off')

def _draw_relative_speedometer(ax, rel_speed_ms):
    ax.clear()
    rel_kmh = float(np.nan_to_num(rel_speed_ms, nan=0.0)) * KMH
    theta = np.linspace(np.pi, 0, 200)
    rel_at_theta = -V_REL_MAX_KMH + 2 * V_REL_MAX_KMH * (np.pi - theta) / np.pi
    for i in range(len(theta) - 1):
        c = _zone_color(rel_at_theta[i], REL_WARN_KMH, REL_LIMIT_KMH)
        ax.plot([np.cos(theta[i]), np.cos(theta[i+1])],
                [np.sin(theta[i]), np.sin(theta[i+1])],
                color=c, linewidth=10, solid_capstyle='butt')
    r_c = max(-V_REL_MAX_KMH, min(rel_kmh, V_REL_MAX_KMH))
    angle = np.pi - np.pi * (r_c + V_REL_MAX_KMH) / (2 * V_REL_MAX_KMH)
    ax.plot([0, 0.85 * np.cos(angle)], [0, 0.85 * np.sin(angle)],
            color='black', linewidth=3)
    ax.scatter([0], [0], s=40, color='black', zorder=5)
    color = _zone_color(rel_kmh, REL_WARN_KMH, REL_LIMIT_KMH)
    label = '--' if np.isnan(rel_speed_ms) else f'{rel_kmh:+.0f}'
    ax.text(0, -0.35, f'rel speed\n{label} km/h', ha='center', va='center',
            fontsize=10, color=color, fontweight='bold')
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.6, 1.2)
    ax.set_aspect('equal'); ax.axis('off')

def _draw_accel_bar(ax, accel_ms2, *, label, amax_g, warn_g, limit_g,
                    neg_text='', pos_text=''):
    ax.clear()
    a_g = float(np.nan_to_num(accel_ms2, nan=0.0)) / G
    a_c = max(-amax_g, min(a_g, amax_g))
    xs = np.linspace(-amax_g, amax_g, 200)
    for i in range(len(xs) - 1):
        c = _zone_color(xs[i], warn_g, limit_g)
        ax.barh(0, xs[i+1] - xs[i], left=xs[i], color=c, alpha=0.22, height=0.6)
    color = _zone_color(a_g, warn_g, limit_g)
    ax.barh(0, a_c, left=0, color=color, height=0.6)
    ax.axvline(0, color='black', linewidth=1)
    val_txt = '--' if np.isnan(accel_ms2) else f'{a_g:+.2f}'
    ax.text(0, -0.55, f'{label}\n{val_txt} g', ha='center', va='center',
            fontsize=10, color=color, fontweight='bold')
    if neg_text:
        ax.text(-amax_g, 0.55, neg_text, ha='left', va='center', fontsize=8, color='#555')
    if pos_text:
        ax.text(amax_g, 0.55, pos_text, ha='right', va='center', fontsize=8, color='#555')
    ax.set_xlim(-amax_g * 1.05, amax_g * 1.05); ax.set_ylim(-1.0, 0.8)
    ax.axis('off')

def _resolve_track(scenario, center_objects_id):
    cid_norm = _normalize_track_id(center_objects_id)
    for tid, t in scenario['tracks'].items():
        if _normalize_track_id(tid) == cid_norm or str(tid) == str(center_objects_id):
            return t
    raise KeyError(f'center track {center_objects_id} not found')

def render_agent_gif_with_dashboard(*, dataset, scenario_id, center_objects_id, split,
                                    output_path, title='', view_radius=70.0,
                                    frame_stride=2, fps=10, max_frames=None):
    import imageio.v2 as imageio
    bundle = load_scenario_by_id(dataset=dataset, scenario_id=scenario_id, split=split)
    scenario = bundle['scenario']
    track = _resolve_track(scenario, center_objects_id)
    positions = np.asarray(track['state']['position'])[:, :2]
    valid = np.asarray(track['state']['valid']).reshape(-1).astype(bool)
    T = len(valid)

    raw_speed, raw_velocity = _per_track_kinematics(positions, valid)
    speed_per_frame_disp    = _smooth(raw_speed,    SPEED_SMOOTH_FRAMES)
    velocity_per_frame_disp = _smooth(raw_velocity, SPEED_SMOOTH_FRAMES)

    accel_vec = _compute_2d_accel(raw_velocity)
    a_long_raw, a_lat_raw = _decompose_long_lat(accel_vec, raw_velocity)
    a_long_per_frame = _smooth(a_long_raw, ACCEL_SMOOTH_FRAMES)
    a_lat_per_frame  = _smooth(a_lat_raw,  ACCEL_SMOOTH_FRAMES)

    rel_per_frame = _smooth(_compute_relative_speed(
        scenario, track, positions, valid,
        speed_per_frame_disp, velocity_per_frame_disp,
    ), SPEED_SMOOTH_FRAMES)

    frames = list(range(0, T, max(1, frame_stride)))
    if max_frames is not None:
        frames = frames[:max_frames]

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig = plt.figure(figsize=(12, 8.5), dpi=110)
    gs = fig.add_gridspec(4, 2, width_ratios=[2.2, 1.0],
                          height_ratios=[1.4, 1.4, 1.0, 1.0],
                          wspace=0.05, hspace=0.30)
    ax_scene = fig.add_subplot(gs[:, 0])
    ax_speed = fig.add_subplot(gs[0, 1])
    ax_rel   = fig.add_subplot(gs[1, 1])
    ax_long  = fig.add_subplot(gs[2, 1])
    ax_lat   = fig.add_subplot(gs[3, 1])

    imgs = []
    for f in frames:
        _draw_scene_frame(ax_scene, scenario, f, str(center_objects_id), view_radius=view_radius)
        ax_scene.set_title(f'{title}  t={f * DT:.1f}s', fontsize=9)
        _draw_speedometer         (ax_speed, speed_per_frame_disp[f] if f < T else np.nan)
        _draw_relative_speedometer(ax_rel,   rel_per_frame[f]        if f < T else np.nan)
        _draw_accel_bar(
            ax_long, a_long_per_frame[f] if f < T else np.nan,
            label='long accel', amax_g=A_LONG_MAX_G,
            warn_g=ACC_LONG_WARN_G, limit_g=ACC_LONG_LIMIT_G,
            neg_text='brake', pos_text='accel',
        )
        _draw_accel_bar(
            ax_lat, a_lat_per_frame[f] if f < T else np.nan,
            label='lat accel', amax_g=A_LAT_MAX_G,
            warn_g=ACC_LAT_WARN_G, limit_g=ACC_LAT_LIMIT_G,
            neg_text='right', pos_text='left',
        )
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0.1)
        buf.seek(0)
        imgs.append(imageio.imread(buf))
    plt.close(fig)
    imageio.mimsave(output_path, imgs, fps=fps, loop=0)
    return output_path

## 4. Render GIFs

In [5]:
CFG = {
    'dataset':      DATASET,
    'split':        SPLIT,
    'label_col':    LABEL_COL,
    'view_radius':  70.0,
    'frame_stride': 2,
    'fps':          10,
    'max_frames':   40,
}

manifest: list[dict] = []
for label, top in picks.items():
    safe_label = str(label).lower().replace(' ', '_')
    out_subdir = GIF_DIR / safe_label
    out_subdir.mkdir(parents=True, exist_ok=True)
    for i, (_, row) in enumerate(top.iterrows()):
        gif_path = out_subdir / f'{safe_label}_tdbm_{DATASET}_scenario={row["scenario_id"]}_center={row["center_objects_id"]}.gif'
        title = (f"TDBM {label} ({DATASET})  margin={row['tdbm_margin']:.2f}  "
                 f"v_avg={row.get('tdbm_v_avg', float('nan')):.1f} "
                 f"j_l={row.get('tdbm_j_l', float('nan')):.2f}")
        try:
            render_agent_gif_with_dashboard(
                dataset=CFG['dataset'], split=CFG['split'],
                scenario_id=str(row['scenario_id']),
                center_objects_id=str(row['center_objects_id']),
                output_path=gif_path,
                title=title,
                view_radius=CFG['view_radius'],
                frame_stride=CFG['frame_stride'],
                fps=CFG['fps'],
                max_frames=CFG['max_frames'],
            )
            manifest.append({'label': str(label), 'rank': i, 'gif': str(gif_path),
                             'scenario_id': str(row['scenario_id']),
                             'center_objects_id': str(row['center_objects_id']),
                             'margin': float(row['tdbm_margin'])})
            print('OK ', gif_path.name)
        except Exception as err:
            print('skip', row['scenario_id'], '->', err)

with open(GIF_DIR / 'manifest.json', 'w') as f:
    json.dump({'config': CFG, 'gifs': manifest}, f, indent=2)
print('\nWrote', len(manifest), 'gifs ->', GIF_DIR)

OK  aggressive_tdbm_av2_scenario=76bfcc82-0598-4e1d-a434-3a2b237813e6_center=10774.gif
OK  aggressive_tdbm_av2_scenario=8628c1e9-f333-4ff2-bf10-f43ad8a5e1a3_center=25759.gif
OK  aggressive_tdbm_av2_scenario=909572a1-9d3d-4917-9297-863506e0f833_center=95250.gif
OK  careful_tdbm_av2_scenario=848f757b-59b7-467b-8c4a-e5c04d8d8e43_center=92990.gif
OK  careful_tdbm_av2_scenario=9beb51ac-620b-45cd-bb07-0acd3579e0e1_center=104696.gif
OK  careful_tdbm_av2_scenario=463bfeb4-a258-4c0e-a789-84ce7284665c_center=47927.gif
OK  careful_tdbm_av2_scenario=66276217-5141-412a-b8b6-0dc471f3bf63_center=141926.gif
OK  careful_tdbm_av2_scenario=9a98713b-137e-4b8d-add9-c79bf23c87f4_center=657.gif
OK  timid_tdbm_av2_scenario=7e821d5b-736d-4ebc-9edd-846fd0799132_center=139053.gif
OK  timid_tdbm_av2_scenario=c25feaae-6370-48e7-9623-7f711b2ac1d2_center=179577.gif
OK  timid_tdbm_av2_scenario=06731c60-8951-4075-8ec8-64cbe8c92e9e_center=18048.gif
OK  timid_tdbm_av2_scenario=5c84bc2f-57d5-4cff-8e20-fb0c543e5298_center

## 5. Inline preview

In [ ]:
from IPython.display import Image, display
for entry in manifest:
    print(f"[{entry['label']}] {entry['gif']}")
    try:
        display(Image(filename=entry['gif']))
    except Exception as err:
        print('  preview failed:', err)